# ClimateAssistant Orchestrator — `openai-gpt-oss-120b`

Runs the full `climdata.LLM` pipeline (Planner → Execution → ClimData → Analytics → Narrator) against the GWDG SAIA OpenAI-compatible API.

**Endpoint:** `https://chat-ai.academiccloud.de/v1`

Export `SAIA_API_KEY` (or `OPENAI_API_KEY`) before running — the key is read from the environment, never stored in the notebook.

The model is set once via `ClimateAssistant(model=...)`, which threads it to both the planner and the narrator.

In [8]:
from climdata.LLM import (
    ClimateAssistant,
    RealClimDataProvider,
    SyntheticProvider,
    build_client,
    print_result,
)

MODEL = "openai-gpt-oss-120b"

In [9]:
import os
from getpass import getpass

# Read the key from the environment; prompt if it isn't set. Keeping it out of
# the cell source means it never lands in the notebook's saved JSON.
api_key = os.environ.get("SAIA_API_KEY") or os.environ.get("OPENAI_API_KEY")
if not api_key:
    api_key = getpass("SAIA API key: ")

client = build_client(api_key=api_key)

## Data provider

`RealClimDataProvider` runs the actual `climdata.ClimData(overrides=...).extract()`. `SyntheticProvider` fabricates a deterministic toy series in memory instead — useful offline, but it never touches the network, so nothing is downloaded. The provider argument is **required**: there is no default, so a run can never silently report fabricated numbers.

`extra_overrides` carries settings the planner never emits — dataset credentials, a non-default `data_dir`. MSWX is fetched from Google Drive and needs a service-account JSON.

Which dataset each role uses now follows: **an explicitly requested dataset wins**, then the role default (`DEFAULT_ROLE_DATASETS`), then auto-resolution by priority. A source that cannot serve a role — an observation product asked to run a scenario — falls back to the role default and records a note in the summary.

In [10]:
MSWX_SERVICE_ACCOUNT = "/home/muduchuru/.climdata_conf/service.json"

extra_overrides = []
if MSWX_SERVICE_ACCOUNT:
    extra_overrides.append(f"+dsinfo.MSWX.params.google_service_account={MSWX_SERVICE_ACCOUNT}")

provider = RealClimDataProvider(extra_overrides=extra_overrides)

# `model=` sets both the planner and the narrator. Use planner_model= /
# narrator_model= to point the two stages at different LLMs.
assistant = ClimateAssistant(llm_client=client, provider=provider, model=MODEL)

## Example 1 — Muncheberg PatchCrop Site (point analysis)

In [12]:
berlin_request = (
    "Describe the historical mean of daily precipitation observations "
    "over Berlin (lat=52.5176, lon=14.1232). Check whether CMIP models capture "
    "the broad climate change signal (difference between future and historical periods). Also describe plausible future climate "
    "conditions for Berlin. Use 2005-2014 as the reference period for the historical mean and 2040-2059 as the future period. "
    "use NASA-POWER for historical data and CMIP6 for future projections. "
)

berlin_result = assistant.run(berlin_request)
print_result(berlin_result)

11:46:46 | INFO    | climdata.orchestrator | ======================================================================
11:46:46 | INFO    | climdata.orchestrator | REQUEST: Describe the historical mean of daily precipitation observations over Berlin (lat=52.5176, lon=14.1232). Check whether CMIP models capture the broad climate change signal (difference between future and historical periods). Also describe plausible future climate conditions for Berlin. Use 2005-2014 as the reference period for the historical mean and 2040-2059 as the future period. use NASA-POWER for historical data and CMIP6 for future projections. 
11:46:52 | INFO    | climdata.orchestrator | stage 'plan' ok (5.89s)
11:46:52 | INFO    | climdata.orchestrator | roles required: ['observation', 'model_historical', 'model_future']
11:46:52 | INFO    | climdata.orchestrator | stage 'execution' ok (0.00s)


✅ All 3652 pr files already exist locally.


11:58:52 | INFO    | climdata.orchestrator | extracted role 'observation' (MSWX)
11:59:00 | INFO    | climdata.orchestrator | extracted role 'model_historical' (CMIP)
11:59:03 | INFO    | climdata.orchestrator | extracted role 'model_future' (CMIP)
11:59:03 | INFO    | climdata.orchestrator | stage 'extract' ok (730.64s)
12:00:46 | INFO    | climdata.orchestrator | stage 'analytics' ok (102.92s)
12:00:49 | INFO    | climdata.orchestrator | stage 'narrate' ok (3.64s)
12:00:49 | INFO    | climdata.orchestrator | DONE: status=ok



########################################################################
STATUS: ok

PLAN: {
  "status": "ready",
  "lat": 52.5176,
  "lon": 14.1232,
  "variable": "pr",
  "time_range": {
    "start": "2005-01-01",
    "end": "2014-12-31"
  },
  "future_time_range": {
    "start": "2040-01-01",
    "end": "2059-12-31"
  },
  "analysis": [
    "climatology",
    "future_projections",
    "model_evaluation"
  ]
}

SUMMARY: {
  "overall_mean": 1.6457,
  "monthly_mean": {
    "1": 1.4298,
    "2": 1.0512,
    "3": 1.0883,
    "4": 0.9167,
    "5": 2.2929,
    "6": 1.7096,
    "7": 3.2175,
    "8": 2.4778,
    "9": 1.6071,
    "10": 1.2169,
    "11": 1.3427,
    "12": 1.3111
  },
  "future_annual_trend": 0.0048,
  "future_summer_trend": -0.0234,
  "projected_mean_change": -0.0699,
  "model_mean": 2.2682,
  "obs_mean": 1.6457,
  "model_bias": 0.6225,
  "rmse": 5.7338,
  "mae": 3.0551,
  "correlation": -0.0082,
  "units": "mm/day"
}

NARRATIVE:
 The long‑term average of the variable is 1.645

## Example 2 — Märkisch-Oderland (area analysis)

In [ ]:
mol_request = (
    "Describe historical and projected daily maximum temperature and precipitation changes over "
    "the Märkisch-Oderland district using the extent lat_min=52.20, lat_max=53.55, "
    "lon_min=13.60, lon_max=15.25. Use NASA POWER for observations and CMIP6 SSP2-4.5 for projections."
)

mol_result = assistant.run(mol_request)
print_result(mol_result)

## Inspecting a single result

Each `PipelineResult` carries the per-stage timings/status, the validated plan, the extracted-data summary, and the grounded narrative separately — useful for debugging which stage degraded if `status != "ok"`.

In [7]:
print("status:", berlin_result.status)
for stage in berlin_result.stages:
    print(f"  {stage.name:10s} {stage.status.value:20s} {stage.seconds:.2f}s")

print("\nnarrative grounded:", berlin_result.provenance.get("narrative_grounded"))
print("\nnarrative:\n", berlin_result.narrative)

status: ok
  plan       ok                   8.15s
  execution  ok                   0.04s
  extract    ok                   724.84s
  analytics  ok                   51.44s
  narrate    ok                   7.34s

narrative grounded: False

narrative:
 The long‑term average of the variable is 1.734 mm/day. Monthly averages display a clear seasonal cycle, with the lowest value in April (0.9035 mm/day) and the highest in July (3.0954 mm/day); the other months range from 1.1782 mm/day in February to 2.3024 mm/day in May. This pattern indicates that the location experiences its greatest daily magnitude in midsummer and its smallest in early spring.

Future projections show an overall increase, with an annual linear trend of 0.0048 mm/day per year, while the summer season is projected to decline, with a linear trend of –0.0234 mm/day per year. The opposite signs of the annual and summer trends suggest that the yearly mean may rise slightly even as summer values are expected to fall.
